# Colab Baseline Runner (Costa)

Run Costa baseline modeling on Colab with DVC artifact pull/repro and DagsHub/MLflow tracking.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%cd /content
!test -d /content/PFE_Experiments/.git || git clone https://github.com/aminetech26/PFE_Experiments.git
%cd /content/PFE_Experiments

/content
fatal: destination path 'PFE_Experiments' already exists and is not an empty directory.
/content/PFE_Experiments


In [11]:
%pip install -q uv
!uv sync

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 316 packages in 1ms
Prepared 233 packages in 1m 44s                                          
Installed 233 packages in 2.23s                             
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0
 + alembic==1.18.4
 + amqp==5.3.1
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.12.1
 + appdirs==1.4.4
 + asyncssh==2.22.0
 + atpublic==7.0.0
 + attrs==25.4.0
 + backoff==2.2.1
 + billiard==4.2.4
 + blinker==1.9.0
 + boto3==1.42.62
 + botocore==1.42.62
 + cachetools==7.0.3
 + catboost==1.2.10
 + celery==5.6.2
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + click-didyoumean==0.3.1
 + click-plugins==1.1.1.2
 + click-repl==0.3.0
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + colorlog==6.10.1
 + configobj==5.0.9
 + contourpy==1.3.3
 + cryptography==46.0.5
 + cuda-bin

In [12]:
# Set these for your session (or use Colab secrets)
import os
os.environ['DAGSHUB_USERNAME'] = 'aminetech26'
os.environ['DAGSHUB_REPO'] = 'PFE_Experiments'
os.environ['DAGSHUB_USER_TOKEN'] = '5e31845f92874871e830dd2f59859e3d632c1aa0'

In [13]:
# Initialize Drive folders used by Optuna persistence
!uv run python -m src.training.colab_drive_init --init

Initialized tracking folders under: /content/drive/MyDrive/PV-FDD/optuna


## DVC-first artifact materialization

`dvc pull` fetches tracked data from DagsHub storage.

If `dvc pull` reports missing cache, make sure the latest `dvc.yaml`, `dvc.lock`, `data/raw.dvc`, and `data/.gitignore` changes have been pushed to GitHub and that DVC artifacts were pushed to DagsHub storage.

If you need a new feature run, edit the `featurize` stage command in `dvc.yaml` first so it matches the exact `task/profile/path` you want, then reproduce the stages below.

In [15]:
!uv run dvc pull

Fetching
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Querying remote cache:   0% 0/5 [00:00<?, ?files/s]
Querying remote cache:   0% 0/5 [00:00<?, ?files/s{'info': ''}]
                                                               
!
  0% |          |0/? [00:00<?,    ?files/s]
md5: 5dcd4d055ded591804a4522cce1e1706.dir
md5: 9cd9dc517d0cfcf39bc90fc23e354cb8.dir
md5: b56562837d4d683dcf76f3daa64330d0.dir
md5: 3e9831b4171fba75b8744e2efc03ef9d.dir
md5: 324116770635e2b4a74b8d4e4bc215c3.dir
Fetching
Building workspace index          |1.00 [00:00,  113entry/s]
Comparing indexes          |17.0 [00:02, 8.17entry/s]
Applying changes          |0.00 [00:00,     ?file/s]
Everything is up to date.
ERROR: failed to pull data from the cloud - Checkout failed for following targets:
data/processed/preprocessed/costa
data/processed/features/costa
experiments/checkpoints/classification/lightgbm_model.pkl
data/raw
data/interim/ingestion/costa
data/interim/splits

In [16]:
# Reproduce Costa pipeline stages when needed
!uv run dvc repro ingest
!uv run dvc repro split
!uv run dvc repro preprocess
!uv run dvc repro featurize

Running stage 'ingest':             
> uv run python -m src.data.ingestion --dataset costa
2026-05-01 14:55:12.418 | INFO     | __main__:<module>:652 - === Stage 0: Data Ingestion — dataset=costa ===
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/PFE_Experiments/src/data/ingestion.py", line 654, in <module>
    df = loader()
         ^^^^^^^^
  File "/content/PFE_Experiments/src/data/ingestion.py", line 285, in load_costa
    raise FileNotFoundError(
FileNotFoundError: Costa .mat file not found: /content/PFE_Experiments/data/raw/Costa PV Fault Dataset/dataset_elec.mat
Clone from https://github.com/clayton-h-costa/pv_fault_dataset into data/raw/Costa PV Fault Dataset/
ERROR: failed to reproduce 'ingest': failed to run: uv run python -m src.data.ingestion --dataset costa, exited with 1
Running stage 'ingest':             
> uv run python -m src.data.ingestion --dataset costa
2026

## Seed policy

Set the seed in one place directly in `configs/model_config.yaml` under `experiment.seed` before running experiments.

## Baseline Commands (per task)

Use one command per task. For each new run, update your choices in `configs/model_config.yaml` (active model, seed, HPO) and ensure the requested `task/profile/split_path` already exists in the generated feature artifacts.

In [17]:
!uv run python -m src.modeling.anomaly_detection.ml.run --task anomaly_semisup --dataset costa --split-path path_a --profile baseline_raw --run-type baseline

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/PFE_Experiments/src/modeling/anomaly_detection/ml/run.py", line 9, in <module>
    from src.modeling.anomaly_detection.ml.matrix_profile_model import run_matrix_profile
  File "/content/PFE_Experiments/src/modeling/anomaly_detection/ml/matrix_profile_model.py", line 9, in <module>
    import matplotlib.pyplot as plt
  File "/content/PFE_Experiments/.venv/lib/python3.12/site-packages/matplotlib/__init__.py", line 1299, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
    ~~~~~~~~^^^^^^^^^^^
  File "/content/PFE_Experiments/.venv/lib/python3.12/site-packages/matplotlib/__init__.py", line 774, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', '